In [ ]:
# Run this cell first to install required packages in JupyterLite
%pip install -q numpy matplotlib ipywidgets scikit-learn


# VAE Latent Space Inference — Section 2.4

**Generation mode** (inference without encoder):
1. Pick any point $z \in \mathbb{R}^2$ in the latent space
2. Pass through decoder: $\hat{x} = p_\phi(x|z)$
3. The output is a generated digit image

Since the VAE learns a **structured prior** $p(z)=\mathcal{N}(0,I)$, nearby points in latent space decode to similar images. This allows smooth interpolation between digit styles.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from sklearn.datasets import load_digits

digits = load_digits()
X = digits.data / 16.0   # (1797, 64)
y = digits.target

def relu(x): return np.maximum(0, x)
def sigmoid(x): return 1/(1+np.exp(-np.clip(x,-15,15)))

# Train a 2D AE with L2 latent regularization
def train_2d_ae(n_epochs=500, lr=5e-3, lam=0.01, seed=42):
    rng = np.random.default_rng(seed)
    def he(fi, sh): return rng.normal(0, np.sqrt(2/fi), sh)
    W_e1=he(64,(64,128)); b_e1=np.zeros(128)
    W_e2=he(128,(128,2)); b_e2=np.zeros(2)
    W_d1=he(2,(2,128));   b_d1=np.zeros(128)
    W_d2=he(128,(128,64));b_d2=np.zeros(64)
    params=[(W_e1,b_e1),(W_e2,b_e2),(W_d1,b_d1),(W_d2,b_d2)]
    ms=[{'W':np.zeros_like(W),'b':np.zeros_like(b)} for W,b in params]
    vs=[{'W':np.zeros_like(W),'b':np.zeros_like(b)} for W,b in params]
    t=0
    def step(W,b,dW,db,m,v,t):
        b1,b2,e=0.9,0.999,1e-8
        m['W']=b1*m['W']+(1-b1)*dW; m['b']=b1*m['b']+(1-b1)*db
        v['W']=b2*v['W']+(1-b2)*dW**2; v['b']=b2*v['b']+(1-b2)*db**2
        W-=lr*(m['W']/(1-b1**t))/(np.sqrt(v['W']/(1-b2**t))+e)
        b-=lr*(m['b']/(1-b1**t))/(np.sqrt(v['b']/(1-b2**t))+e)
    losses=[]
    for epoch in range(n_epochs):
        idx=rng.permutation(len(X))
        for s in range(0,len(X),64):
            xb=X[idx[s:s+64]]; n=len(xb); t+=1
            h1=relu(xb@W_e1+b_e1); z=h1@W_e2+b_e2
            hd=relu(z@W_d1+b_d1); xhat=sigmoid(hd@W_d2+b_d2)
            dxhat=-2*(xb-xhat)/n
            s_out=sigmoid(hd@W_d2+b_d2); ds=dxhat*s_out*(1-s_out)
            dWd2=hd.T@ds; dbd2=ds.sum(0)
            dhd=(ds@W_d2.T)*(hd>0); dWd1=z.T@dhd; dbd1=dhd.sum(0)
            dz=dhd@W_d1.T+lam*2*z/n; dWe2=h1.T@dz; dbe2=dz.sum(0)
            dh1=(dz@W_e2.T)*(h1>0); dWe1=xb.T@dh1; dbe1=dh1.sum(0)
            for (W,b),(dW,db),m,v in zip(params,[(dWe1,dbe1),(dWe2,dbe2),(dWd1,dbd1),(dWd2,dbd2)],ms,vs):
                step(W,b,dW,db,m,v,t)
        if epoch%100==99:
            z_a=relu(X@W_e1+b_e1)@W_e2+b_e2
            xh=sigmoid(relu(z_a@W_d1+b_d1)@W_d2+b_d2)
            losses.append(np.mean((X-xh)**2))
    z_final=relu(X@W_e1+b_e1)@W_e2+b_e2
    return (W_d1,b_d1,W_d2,b_d2), z_final, losses

print('Training 2D AE on digits (500 epochs)...')
(W_d1,b_d1,W_d2,b_d2), z_all, losses = train_2d_ae()
print(f'Done. Final MSE={losses[-1]:.4f}')


def decode_z(z1, z2):
    z = np.array([[z1, z2]])
    h = relu(z @ W_d1 + b_d1)
    return sigmoid(h @ W_d2 + b_d2).reshape(8, 8)


def draw_inference(z1=0.0, z2=0.0, show_interp=True):
    img = decode_z(z1, z2)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # 1. Latent space scatter
    colors = plt.cm.tab10(np.linspace(0,1,10))
    for d in range(10):
        mask = y==d
        axes[0].scatter(z_all[mask,0], z_all[mask,1], c=[colors[d]], s=8, alpha=0.5, label=str(d))
    axes[0].scatter([z1],[z2], c='black', s=120, marker='x', linewidths=3, zorder=10, label='Selected z')
    axes[0].set_xlabel('z₁'); axes[0].set_ylabel('z₂')
    axes[0].set_title('2D Latent Space', fontsize=11)
    axes[0].legend(fontsize=7, ncol=5, loc='upper right')
    axes[0].grid(True, alpha=0.3)

    # 2. Generated digit
    axes[1].imshow(img, cmap='gray_r', vmin=0, vmax=1)
    axes[1].set_xticks([]); axes[1].set_yticks([])
    axes[1].set_title(f'Generated digit\nz=({z1:.2f},{z2:.2f})', fontsize=11)

    # 3. Interpolation or grid
    if show_interp:
        # Interpolate from (z1,z2) to (−z1,−z2)
        ts = np.linspace(0, 1, 9)
        imgs = [decode_z(z1*(1-t)+(-z1*0.5)*t, z2*(1-t)+(-z2*0.5)*t) for t in ts]
        strip = np.hstack(imgs)
        axes[2].imshow(strip, cmap='gray_r', vmin=0, vmax=1, aspect='auto')
        axes[2].set_xticks([]); axes[2].set_yticks([])
        axes[2].set_title('Interpolation: z → (0,0) in 9 steps', fontsize=10)
    else:
        z_grid = np.linspace(-3,3,8)
        grid = np.block([[decode_z(z,w) for z in z_grid] for w in reversed(z_grid)])
        axes[2].imshow(grid, cmap='gray_r', vmin=0, vmax=1)
        axes[2].set_xticks([]); axes[2].set_yticks([])
        axes[2].set_title('8×8 grid of decoded latent points', fontsize=10)

    plt.tight_layout(); plt.show()


widgets.interact(draw_inference,
    z1=widgets.FloatSlider(min=-4.5, max=3.5, step=0.1, value=0.0, description='z₁', continuous_update=False),
    z2=widgets.FloatSlider(min=-5.0, max=3.0, step=0.1, value=0.0, description='z₂', continuous_update=False),
    show_interp=widgets.Checkbox(value=True, description='Interpolation (vs grid)'),
)